### Проверка, что S3 и Spark работают

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("SCD2_Homework")
         .master("spark://spark-master:7077")
         .getOrCreate())

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
data_test = [("test_user", 1), ("another_user", 2)]
df_test = spark.createDataFrame(data_test, ["name", "value"])
df_test.write.mode("overwrite").parquet("s3a://dwh/_test/test.parquet")

df_read = spark.read.parquet("s3a://dwh/_test/test.parquet")
df_read.show()

+------------+-----+
|        name|value|
+------------+-----+
|another_user|    2|
|   test_user|    1|
+------------+-----+



### Генерация исходных данных

In [3]:
%%bash
python3 generate_event_log.py \
  --rows 5000000 \
  --users 100000 \
  --change-probability 0.20 \
  --seed 42 \
  --start-time 2026-01-01T00:00:00Z \
  --max-step-seconds 1 \
  --user-id-start 1 \
  --rows-per-file 100000 \
  --output-dir event_log_22

files=50/50 rows=5000000/5000000 rate=137,050 rows/s


Готово: 5000000 строк в каталоге event_log_22


### Приведение к SCD2 (ctrl+c из дз2_1)

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# задаём всё время в формате UTC
spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

source_path = "s3a://dwh/source/event_log/"

# читаем исходные данные
df_raw = (
    spark.read
    .option("header", "true")
    .option("mode", "FAILFAST")
    .csv(source_path)
)

# приводим типы
df_events = df_raw.select(
    F.col("event_time").alias("_event_time_raw"),
    F.to_timestamp(
        F.trim(F.col("event_time")),
        "yyyy-MM-dd'T'HH:mm:ss'Z'"
    ).alias("event_time"),
    F.col("user_id").alias("_user_id_raw"),
    F.col("user_id").cast("long").alias("user_id"),
    F.col("user_fio").cast("string").alias("user_fio"),
    F.col("user_email").cast("string").alias("user_email"),
    F.col("user_login").cast("string").alias("user_login")
)

window_user = Window.partitionBy("user_id").orderBy("event_time")
df_with_profile = (
    df_events
    .withColumn(
        "_profile",
        F.struct("user_fio", "user_email", "user_login")
    )
    .withColumn(
        "_previous_profile",
        F.lag("_profile").over(window_user)
    )
)
df_changed = df_with_profile.filter(
    F.col("_previous_profile").isNull() |
    ~F.col("_profile").eqNullSafe(F.col("_previous_profile"))
)
df_versions = df_changed.select(
    "user_id",
    "user_fio",
    "user_email",
    "user_login",
    F.col("event_time").alias("valid_from")
)

window_versions = Window.partitionBy("user_id").orderBy("valid_from")
df_scd2 = (
    df_versions
    .withColumn("valid_to", F.lead("valid_from").over(window_versions))
    .withColumn("is_current", F.col("valid_to").isNull())
)

df_scd2.orderBy(
    "user_id",
    "valid_from"
).show(30, truncate=False)

+-------+---------------+----------------------------------+----------+-------------------+-------------------+----------+
|user_id|user_fio       |user_email                        |user_login|valid_from         |valid_to           |is_current|
+-------+---------------+----------------------------------+----------+-------------------+-------------------+----------+
|1      |John Scott     |john.scott.1@post.example         |jscott1   |2026-01-01 06:28:21|2026-01-02 12:36:10|false     |
|1      |John Scott     |john.scott.1@post.example         |jscott1r1 |2026-01-02 12:36:10|2026-01-04 06:56:45|false     |
|1      |Brittany Reed  |john.scott.1@post.example         |jscott1r1 |2026-01-04 06:56:45|2026-01-22 18:33:05|false     |
|1      |Brittany Reed  |brittany.reed.1.r3@webmail.example|jscott1r1 |2026-01-22 18:33:05|2026-01-27 18:39:48|false     |
|1      |Justin Wilson  |brittany.reed.1.r3@webmail.example|jscott1r1 |2026-01-27 18:39:48|2026-02-03 23:38:52|false     |
|1      |Justin 

### Сохранение в S3
Сохраняем в S3 так как все задания будут выполняться в отдельных ноутбуках

In [4]:
target_path = "s3a://dwh/scd2/user_events/"
df_scd2.write.mode("overwrite").parquet(target_path)

df_scd2_result = spark.read.parquet(target_path)
df_scd2_result.printSchema()
df_scd2_result.orderBy(
    "user_id",
    "valid_from"
).show(30, truncate=False)

root
 |-- user_id: long (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- valid_to: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)

+-------+---------------+----------------------------------+----------+-------------------+-------------------+----------+
|user_id|user_fio       |user_email                        |user_login|valid_from         |valid_to           |is_current|
+-------+---------------+----------------------------------+----------+-------------------+-------------------+----------+
|1      |John Scott     |john.scott.1@post.example         |jscott1   |2026-01-01 06:28:21|2026-01-02 12:36:10|false     |
|1      |John Scott     |john.scott.1@post.example         |jscott1r1 |2026-01-02 12:36:10|2026-01-04 06:56:45|false     |
|1      |Brittany Reed  |john.scott.1@post.example         |jscott1r1 |2026-01-04 06:56